# 17_dude_workflow — DUD-E decoy 전체 워크플로

active 전체를 DUD-E에 넣어 decoy(가정 inactive)를 대량 생성하고, 받은 decoy를 정제해
1:1 / 1:5 / 1:10 학습셋까지 만든다.

**3부 구성 (PART A는 지금, B·C는 decoy 받은 뒤):**
1. **PART A** — 모든 active를 45개씩 배치 파일로 생성 -> DUD-E에 제출
2. **PART B** — 받은 decoy를 중복제거(문자열->canonical->InChIKey) + active/inactive 겹침 제거 + Tanimoto 검증 -> 한 파일로 병합
3. **PART C** — 랜덤 샘플로 1:1 / 1:5 / 1:10 학습셋 구축

> 아래 '공용 함수'까지는 항상 먼저 실행하세요.

In [ ]:
# 프로젝트 루트로 이동 + import
import os
if not os.path.isdir('data') and os.path.basename(os.getcwd()) in ('notebooks', 'scripts'):
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import glob, math, random
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdFingerprintGenerator, DataStructs
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')
NL = chr(10)

### 공용 함수 (분류·정제·fingerprint)

In [ ]:
# 공용 함수: 분류 규칙 + SMILES 정제(염 제거 -> canonical + InChIKey) + fingerprint
def klass(v, rel):
    rel = str(rel).strip()
    if rel in ('<', '<='):
        return 'active' if v <= 10000 else ('gray' if v < 20000 else 'inactive')
    if rel in ('>', '>='):
        return 'inactive' if v >= 20000 else ('gray' if v > 10000 else 'amb')
    return 'active' if v <= 10000 else ('gray' if v < 20000 else 'inactive')

_lfc = rdMolStandardize.LargestFragmentChooser()
def clean(smi):
    m = Chem.MolFromSmiles(str(smi))
    if m is None:
        return None
    m = _lfc.choose(m)                 # 염/부성분 제거 -> 가장 큰 조각
    try:
        ik = Chem.MolToInchiKey(m)     # InChIKey (호변이성질체 정규화)
    except Exception:
        return None
    if not ik:
        return None
    return Chem.MolToSmiles(m), ik, Descriptors.MolWt(m)

_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
def fp_of(smi):
    m = Chem.MolFromSmiles(str(smi))
    return _gen.GetFingerprint(m) if m else None

---
## PART A — DUD-E 입력 생성 (지금 실행)
모든 active를 InChIKey로 중복제거하고 45개씩 배치로 나눠 저장한다.

### A-파라미터

In [ ]:
# ===== PART A 파라미터 =====
SRC = 'data/HSD17B13_IC50_merged.xlsx'
N_PER_BATCH = 45          # DUD-E 1회 제출 개수(상한 50)
MW_MAX = None             # None = 상한 없음(모든 active). 제한하려면 700 등으로
CURATED = 'data/curated_ligands.csv'   # 정제된 active/inactive 목록
BATCH_DIR = 'data/dude_input'          # 배치 .smi 저장 폴더
os.makedirs(BATCH_DIR, exist_ok=True)

### A-1. 큐레이션 (gray·애매 `>` 제거, InChIKey 중복제거)

In [ ]:
# 큐레이션: 분류 -> InChIKey 중복제거 -> active/inactive만(gray,amb 제거) -> 저장
df = pd.read_excel(SRC, sheet_name='same_dedup_keepdiff').dropna(subset=['canonical_smiles', 'ic50_nM'])
rank = {'active': 0, 'gray': 1, 'inactive': 2, 'amb': 3}   # 하나라도 active면 active 우선
best = {}
for smi, v, rel in zip(df.canonical_smiles, df.ic50_nM, df.relation):
    r = clean(smi)
    if r is None:
        continue
    cs, ik, mw = r
    c = klass(v, rel)
    if ik not in best or rank[c] < best[ik][0]:
        best[ik] = (rank[c], cs, mw, c)

rows = [(cs, ik, mw, c) for ik, (p, cs, mw, c) in best.items() if c in ('active', 'inactive')]
cur = pd.DataFrame(rows, columns=['canonical_smiles', 'inchikey', 'mw', 'label'])
cur.to_csv(CURATED, index=False)
print('정제 결과:', dict(cur.label.value_counts()))
print('저장:', CURATED)

### A-2. 배치 파일 생성 (모든 active, 45개씩)

In [ ]:
# 모든 active를 45개씩 배치로 나눠 DUD-E 입력 파일 생성 (랜덤 셔플 후 청크)
acts = cur[cur.label == 'active'].copy()
if MW_MAX is not None:
    acts = acts[acts.mw <= MW_MAX]
smis = acts.canonical_smiles.tolist()
random.seed(42)
random.shuffle(smis)
nbatch = math.ceil(len(smis) / N_PER_BATCH)
for b in range(nbatch):
    chunk = smis[b * N_PER_BATCH:(b + 1) * N_PER_BATCH]
    fn = os.path.join(BATCH_DIR, 'batch%02d.smi' % (b + 1))
    with open(fn, 'w') as f:
        for j, s in enumerate(chunk, 1):
            f.write(s + ' b%02d_act_%02d' % (b + 1, j) + NL)
print('active', len(smis), '개 ->', nbatch, '배치 생성 ->', BATCH_DIR + '/batch01.smi ...')
print('각 batchNN.smi를 DUD-E에 제출하세요 (1회 ~10분, 총 %d회).' % nbatch)

---
## PART B — decoy 후처리 (decoy 받은 뒤 실행)

DUD-E 결과(`.picked`)를 `data/dude_decoys/` 아래에 **배치 폴더째** 넣고 실행. 로더가 하위폴더를
재귀 탐색하고 각 `.picked`의 첫 줄('ligand'=active)은 건너뛴다. 배치가 달라도 decoy가 겹치므로
문자열->canonical->InChIKey 3단계로 중복제거하고, active/기존 inactive와 안 겹치는지,
active와 구조가 충분히 다른지(Tanimoto) 검증한다.

### B-파라미터

In [ ]:
# ===== PART B 파라미터 (decoy 받은 뒤 실행) =====
DECOY_DIR = 'data/dude_decoys'    # DUD-E .picked 결과를 하위폴더째 여기에 (배치별 폴더 OK)
SIM_MAX = 0.35                     # active와 ECFP4 Tanimoto 이 값 초과면 제외(구조 충분히 다른지)
DECOY_OUT = 'data/decoys_clean.csv'
os.makedirs(DECOY_DIR, exist_ok=True)

### B-1. decoy 파일 로드 (하위폴더 재귀, .picked)

In [ ]:
# decoy 로드: 하위폴더 재귀 + DUD-E .picked 형식(탭구분, 첫 줄 'ligand'=active는 제외)
files = []
for root, _dirs, fnames in os.walk(DECOY_DIR):
    for fn in fnames:
        files.append(os.path.join(root, fn))
print('decoy 파일', len(files), '개 (하위 폴더 포함)')
raw = []
for fn in files:
    try:
        fh = open(fn)
    except Exception:
        continue
    for line in fh:
        parts = line.split()                     # 탭/공백 분리
        if not parts:
            continue
        if parts[0] == 'ligand':                 # .picked 첫 줄 = active(쿼리) -> 제외
            continue
        tok = parts[0]
        if tok.lower() in ('smiles', 'canonical_smiles', 'smi'):
            continue
        raw.append(tok)
    fh.close()
print('원시 decoy 줄 수:', len(raw))
if not raw:
    print('[대기] decoy가 없습니다. DUD-E .picked 결과를', DECOY_DIR, '에 넣고 다시 실행하세요.')

### B-2. 3단계 중복제거 + 겹침 제거 + Tanimoto 검증 + 병합 저장

In [ ]:
# 정제 + 3단계 중복제거(문자열->canonical->InChIKey) + active/inactive 겹침 + Tanimoto 검증
cur = pd.read_csv(CURATED)
act = cur[cur.label == 'active']
act_ik = set(act.inchikey)
ina_ik = set(cur[cur.label == 'inactive'].inchikey)
act_fps = [f for f in (fp_of(s) for s in act.canonical_smiles) if f is not None]

# 1차: 원시 문자열 중복 제거 (DUD-E는 같은 decoy를 여러 active에 재사용 -> 대량 중복)
raw_u = list(dict.fromkeys(raw))
print('원시', len(raw), '-> 문자열 유일', len(raw_u))

# 2차: canonical 정규화 + canonical 중복 제거 (InChI는 비싸므로 여기서 먼저 줄임)
canon_seen, canon_list = set(), []
for smi in raw_u:
    m = Chem.MolFromSmiles(smi)
    if m is None:
        continue
    m = _lfc.choose(m)
    cs = Chem.MolToSmiles(m)
    if cs not in canon_seen:
        canon_seen.add(cs)
        canon_list.append(cs)
print('canonical 유일', len(canon_list))

# 3차: InChIKey 중복 + active/inactive 겹침 + Tanimoto 검증
seen_ik, kept = set(), []
n_dup = n_oa = n_oi = n_sim = 0
for i, cs in enumerate(canon_list):
    m = Chem.MolFromSmiles(cs)
    try:
        ik = Chem.MolToInchiKey(m)
    except Exception:
        continue
    if not ik or ik in seen_ik:
        n_dup += 1; continue
    seen_ik.add(ik)
    if ik in act_ik:
        n_oa += 1; continue
    if ik in ina_ik:
        n_oi += 1; continue
    f = _gen.GetFingerprint(m)
    sim = max(DataStructs.BulkTanimotoSimilarity(f, act_fps)) if act_fps else 0.0
    if sim > SIM_MAX:
        n_sim += 1; continue
    kept.append((cs, ik, Descriptors.MolWt(m), round(float(sim), 3)))
    if (i + 1) % 20000 == 0:
        print('  처리', i + 1, '/', len(canon_list), '| 현재 clean', len(kept))

print('InChIKey중복 %d | active겹침 %d | inactive겹침 %d | 유사(>%.2f) %d 제거'
      % (n_dup, n_oa, n_oi, SIM_MAX, n_sim))
dec = pd.DataFrame(kept, columns=['canonical_smiles', 'inchikey', 'mw', 'max_sim_active'])
dec.to_csv(DECOY_OUT, index=False)
try:
    dec.to_excel(DECOY_OUT.replace('.csv', '.xlsx'), index=False)
except Exception:
    pass
print('최종 clean decoy:', len(dec), '개 ->', DECOY_OUT)
if len(dec):
    print('active와 최대유사도: 중앙값 %.2f, 최대 %.2f (SIM_MAX %.2f 이하 보장)'
          % (dec.max_sim_active.median(), dec.max_sim_active.max(), SIM_MAX))

---
## PART C — 학습셋 구축 (1:1 / 1:5 / 1:10)

실측 inactive는 전부 포함하고, 나머지는 clean decoy에서 **랜덤 샘플**해 비율을 맞춘다
(구조 편향 방지). active:inactive = 1:1, 1:5, 1:10 세 세트를 각각 CSV로 저장.

In [ ]:
# ===== PART C: 1:1 / 1:5 / 1:10 학습셋 구축 (랜덤 샘플로 편향 방지) =====
RATIOS = [1, 5, 10]
cur = pd.read_csv(CURATED)
act = cur[cur.label == 'active'][['canonical_smiles', 'inchikey']].copy()
ina = cur[cur.label == 'inactive'][['canonical_smiles', 'inchikey']].copy()
dec = (pd.read_csv(DECOY_OUT)[['canonical_smiles', 'inchikey']]
       if os.path.exists(DECOY_OUT) else pd.DataFrame(columns=['canonical_smiles', 'inchikey']))

n_act = len(act)
print('active %d | 실측 inactive %d | clean decoy %d' % (n_act, len(ina), len(dec)))
for r in RATIOS:
    need = n_act * r
    need_dec = max(0, need - len(ina))               # 실측 inactive는 전부 포함, 나머지는 decoy
    if need_dec > len(dec):
        print('[1:%d] decoy 부족: 필요 %d, 보유 %d -> 가능한 만큼만' % (r, need_dec, len(dec)))
        samp = dec
    else:
        samp = dec.sample(n=need_dec, random_state=r)  # 랜덤 샘플(편향 방지)
    inact = pd.concat([ina.assign(source='real_inactive'),
                       samp.assign(source='decoy')], ignore_index=True)
    train = pd.concat([act.assign(label=1, source='active'),
                       inact.assign(label=0)], ignore_index=True)
    train = train.sample(frac=1, random_state=r).reset_index(drop=True)   # 전체 셔플
    out = 'data/train_1to%d.csv' % r
    train[['canonical_smiles', 'inchikey', 'label', 'source']].to_csv(out, index=False)
    print('[1:%d] active %d : inactive %d  -> %s' % (r, n_act, len(inact), out))